# Quick Run: AU Corrections Harness

Execute all four correction steps and view results inline.

In [ ]:
# Setup
import os, subprocess, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Clone and setup
repo = Path("/content/AIB")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/samiraghafarigousheh-sys/aib.git", str(repo)], check=True, capture_output=True)

os.chdir(repo)

# Install
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "pandas"], capture_output=True)

print("✓ Setup complete")

In [ ]:
# Run harness
print("Running comparison harness (this takes ~5-10 minutes)...\n")
result = subprocess.run(
    [sys.executable, "examples/compare_au_corrections.py", "--through-step", "4"],
    capture_output=True, text=True, timeout=3600
)

if result.returncode == 0:
    print("✓ Harness completed\n")
else:
    print(f"✗ Failed (code {result.returncode})")
    print(result.stderr[-500:])

In [ ]:
# Load and display results
df = pd.read_csv(repo / "results" / "au_corrections_summary" / "comparison.csv")
print("\n" + "="*90)
print("COMPARISON: States × Metrics (Heating, Cooling, Total in kWh/m²)")
print("="*90)
print(df.to_string(index=False))
print()

# Calculate reductions
baseline = df[df["State"] == "Baseline"].iloc[0]["Total (kWh/m²)"]
final = df[df["State"].str.contains("Hemisphere")].iloc[0]["Total (kWh/m²)"]
reduction = baseline - final
pct_reduction = 100 * reduction / baseline

print(f"Total Reduction: {baseline:.1f} → {final:.1f} kWh/m² (−{pct_reduction:.1f}%)")
print("="*90)

In [ ]:
# Display chart
df_by_metric = pd.read_csv(repo / "results" / "au_corrections_summary" / "comparison_by_metric.csv")

# Prepare data for grouped bar chart
metrics = df_by_metric["Metric"].tolist()
states = [c for c in df_by_metric.columns if c != "Metric"]

fig, ax = plt.subplots(figsize=(14, 7))
x = range(len(metrics))
width = 0.14

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

for i, state in enumerate(states):
    values = pd.to_numeric(df_by_metric[state], errors='coerce')
    ax.bar([xi + width*(i-len(states)/2) for xi in x], values, width, label=state, color=colors[i % len(colors)])

ax.set_xlabel("Metric", fontsize=12, fontweight='bold')
ax.set_ylabel("Energy (kWh/m²·yr)", fontsize=12, fontweight='bold')
ax.set_title("Australian Building Energy Corrections: Cumulative Impact", fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, rotation=0)
ax.legend(loc='upper right', fontsize=10, ncol=2)
ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

print("\n✓ Chart displayed")

## Summary

**Step 1 — Internal Gains:** Remove 7.335× inflation in adjacent-zone gains  
**Step 2 — Conditioned Zones:** Hold neighbours at 20°C setpoint instead of buffer formula  
**Step 3 — Ground Contact:** Fix fallback that assigned full footprint to untagged buildings  
**Step 4 — Hemisphere:** Resolve coldest_month from latitude (July for south, January for north)  

**Result:** Energy intensity 172.9 → 34.9 kWh/m²·yr (−79.8%)

### Full Documentation

See `results/step_*/NOTES.md` for detailed analysis, or view the comprehensive Jupyter notebook `colab_au_corrections_harness.ipynb` for more explanation.